# Assignment 10 - Problem 2
## DeepSeek-R1 on my own computer with Ollama

**Name:** Hussein Jawad

For this problem I installed Ollama on my PC, downloaded the DeepSeek-R1 model, checked with `ollama ps` that it was running, and then used `ollama.chat()` in Python to get an answer together with the model's chain of thought.

My question for the model was: "Explain the difference between the General and Special theory of relativity."

Steps:
1. Install Ollama
2. Check my machine and pick the model size
3. Pull the model
4. Run the model and check it with `ollama ps`
5. Ask the question with `ollama.chat()`

At the end are my observations and an appendix with the concepts.

## Step 1 - Install Ollama

I downloaded Ollama for Windows from https://ollama.com/download and installed it. It runs in the background as a local server at `http://localhost:11434`, and both the `ollama` command and the Python library connect to it.

I also installed the `ollama` Python library (for `ollama.chat()`) and `psutil` to check my memory.

In [1]:
# Install the Ollama Python library (for ollama.chat) and psutil (to read the machine's memory)
%pip install -q ollama psutil

Note: you may need to restart the kernel to use updated packages.


Checking that Ollama is installed (the `!` runs a terminal command from the notebook):

In [2]:
# Show the installed Ollama version
!ollama --version

ollama version is 0.40.16


## Step 2 - Check my machine and pick the model size

DeepSeek-R1 comes in different sizes (1.5b, 7b, 8b, 14b, 32b, 70b and 671b, where b means billion parameters). The lab says you need about as much memory as the download size, plus some extra. The 8b model is a 5.2 GB download, so a PC with 8 GB of RAM or more should be able to run it.

In [3]:
import platform, shutil, subprocess   # operating system info, find programs, run commands
import psutil                         # memory info

print("Operating system:", platform.system(), platform.release())                 # e.g. Windows 11
print("CPU cores       :", psutil.cpu_count(logical=True))                         # number of CPU threads
print("RAM total       :", round(psutil.virtual_memory().total / 1e9, 1), "GB")    # installed memory
print("RAM available   :", round(psutil.virtual_memory().available / 1e9, 1), "GB")  # memory free right now

if shutil.which("nvidia-smi"):                                                       # an NVIDIA GPU driver is installed
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()             # ask the driver for name and memory
    print("GPU             :", gpu)                                                 # e.g. NVIDIA L4, 23034 MiB
else:
    print("GPU             : no NVIDIA GPU found (the model will run on the CPU, which is slower)")

Operating system: Windows 11
CPU cores       : 12
RAM total       : 16.8 GB
RAM available   : 2.0 GB
GPU             : no NVIDIA GPU found (the model will run on the CPU, which is slower)


My PC has 16.8 GB of RAM and no NVIDIA GPU, so the model has to run on the CPU. 16.8 GB is enough for the 8b model, which is also the size the assignment suggests, so I went with `deepseek-r1:8b`. The bigger ones (14b is about 9 GB, 32b about 20 GB) would be too slow on a CPU or wouldn't fit. Only 2.0 GB was free when I ran this because I had other programs open, but the model still loaded fine in Step 4.

In [4]:
MODEL = "deepseek-r1:8b"   # 8B is the size the assignment recommends; use "deepseek-r1:1.5b" if memory is low
print("Model:", MODEL)     # the model used in the rest of the notebook

Model: deepseek-r1:8b


## Step 3 - Pull the model

`ollama.pull()` does the same thing as `ollama pull deepseek-r1:8b` in the terminal. The first time it downloads about 5.2 GB, and after that it just checks that the model is there. I print each stage only once so the output stays short.

In [5]:
import ollama   # Ollama Python library: talks to the Ollama server on localhost:11434

last_status = None                                  # last stage printed
for progress in ollama.pull(MODEL, stream=True):    # download the model, receiving progress updates
    if progress.status != last_status:              # print each new stage only once
        print(progress.status)
        last_status = progress.status

pulling manifest
pulling e6a7edc1a4d7
pulling c5ad996bda6e
pulling 6e4c38e1172f
pulling ed8474dc73db
pulling f64cd5418e4b
verifying sha256 digest
writing manifest
removing unused layers
success


## Step 4 - Run the model and check it with `ollama ps`

To load the model without asking it anything, I send an empty prompt. I set `keep_alive="30m"` so it stays loaded for 30 minutes instead of the default 5, which means it is still in memory for Step 5. Then I run `ollama ps` to see what is running.

In [7]:
import time                                                     # to measure how long things take

start = time.time()                                             # start the timer
ollama.generate(model=MODEL, prompt="", keep_alive="30m")       # empty prompt = just load the model into memory
print(f"Model loaded in {time.time() - start:.1f} seconds")     # loading time

# Show the models that are running right now
!ollama ps

Model loaded in 17.2 seconds
NAME              ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL               
deepseek-r1:8b    6995872bfe4c    5.9 GB    100% CPU     4096       llamacpp    29 minutes from now    


`ollama ps` shows that `deepseek-r1:8b` is running. It uses 5.9 GB of memory, which is more than the 5.2 GB download because the context also needs memory. PROCESSOR says 100% CPU because I don't have an NVIDIA GPU. The context is 4096 tokens, and the model stays loaded for another 29 minutes because of `keep_alive`.

## Step 5 - Ask the question with `ollama.chat()`

`ollama.chat()` takes the model name and a list of messages. With `think=True`, DeepSeek-R1 gives back its reasoning in a separate field:
- `response.message.thinking` is the chain of thought
- `response.message.content` is the final answer

The model writes all of its thinking before it answers, so this takes a while. On my PC (no GPU) it took about 16 minutes.

In [9]:
QUESTION = "Explain the difference between the General and Special theory of relativity."   # my question

start = time.time()                                          # start the timer
response = ollama.chat(
    model=MODEL,                                             # deepseek-r1:8b
    messages=[{"role": "user", "content": QUESTION}],       # one user message
    think=True,                                              # return the chain of thought separately
    keep_alive="30m",                                        # keep the model loaded afterwards
)
seconds = time.time() - start                                # how long the whole call took

thinking = response.message.thinking or ""                   # chain of thought (empty if the model gave none)
answer = response.message.content                            # final answer

print(f"Time                : {seconds:.1f} s")                                        # wall-clock time
print(f"Prompt tokens       : {response.prompt_eval_count}")                           # tokens in my question
print(f"Generated tokens    : {response.eval_count}")                                  # thinking + answer tokens
print(f"Speed               : {response.eval_count / (response.eval_duration / 1e9):.1f} tokens/s")   # durations are in ns
print(f"Chain of thought    : {len(thinking):,} characters")                           # length of the thinking
print(f"Final answer        : {len(answer):,} characters")                             # length of the answer
print(f"Thinking kept apart : {'<think>' not in answer and len(thinking) > 0}")       # True = two separate fields

Time                : 981.5 s
Prompt tokens       : 17
Generated tokens    : 2193
Speed               : 2.3 tokens/s
Chain of thought    : 4,174 characters
Final answer        : 6,366 characters
Thinking kept apart : True


### The chain of thought

The thinking is pretty long, so I only print the start and the end.

In [10]:
if len(thinking) > 2500:                                      # long: show the start and the end only
    print(thinking[:1800])                                    # first 1,800 characters
    print(f"\n[... {len(thinking) - 2500:,} characters skipped ...]\n")
    print(thinking[-700:])                                    # last 700 characters
else:
    print(thinking)                                           # short enough to show in full


Okay, the user asked to explain the difference between General and Special Relativity. Let me start by recalling what I know. Special Relativity is about constant speed, no gravity, while General adds gravity and acceleration.

Hmm, the user might be a student studying physics, or maybe someone curious about these concepts. They probably have a basic understanding but need a clear distinction. I should break down the key points without too much jargon.

First, I need to mention the two postulates of Special Relativity. The invariance of the speed of light and the principle of relativity. That's foundational. Then, the main effects like time dilation and length contraction. Maybe include examples like muons or light travel to make it relatable.

Wait, the user might not know what postulates are, so I should explain them simply. Also, they might not grasp why the speed of light is a universal constant. Emphasize that it's the same for all observers, which is counterintuitive.

Now, Gene

### The final answer

In [11]:
print(answer)   # the model's final answer, without the thinking


Okay, let's break down the difference between Special Relativity (SR) and General Relativity (GR). Einstein developed SR in 1905 and GR in 1915, with GR building upon the foundations of SR.

Here's the comparison:

## Special Relativity (1905)

*   **Scope:** Deals with physics in **inertial reference frames** (frames moving at constant velocity, not accelerating). It does not include gravity.
*   **Core Idea:** The laws of physics are the same for all observers in uniform motion (constant velocity), and the speed of light in a vacuum is constant for all observers, regardless of the motion of the light source or observer.
*   **Key Postulates:**
    1.  The principle of relativity: The laws of physics are identical in all inertial reference frames.
    2.  The invariance of the speed of light: The speed of light in a vacuum (c) is the same for all observers, even those moving relative to each other.
*   **Main Effects:** Focuses on how space and time are perceived differently by obser

## Observations

1. Ollama 0.40.16 is installed on my Windows 11 PC, and the pull finished with `success`, so `deepseek-r1:8b` (5.2 GB) is on my computer.

2. The model is running. It took 17.2 seconds to load, and `ollama ps` showed it with 5.9 GB in memory and 100% CPU.

3. It was very slow. The answer took 981.5 seconds, about 16 minutes. The model generated 2,193 tokens at only 2.3 tokens per second, so almost all of that time was spent generating text. I think the main reasons are that it runs only on my CPU, and that a reasoning model writes a lot of thinking before the actual answer. With a GPU, or with the smaller 1.5b model, it would be a lot faster. I also only had 2.0 GB of RAM free, so closing other programs first might help a bit.

4. The chain of thought really came back separately from the answer: 4,174 characters of thinking and 6,366 characters of answer, and the check printed `True` (no `<think>` tags in the answer). When I read the thinking, it is basically the model planning its answer. It starts with what it knows (special relativity is about constant speed with no gravity, general relativity adds gravity and acceleration), guesses that I'm probably a student, and decides to explain each theory first and then compare them. The final answer follows that plan. One thing I noticed is that it talks about using muons, an elevator and a trampoline as examples, but none of those show up in the final answer. So the thinking is more like a rough plan than an exact draft.

5. I compared the answer with the summary in the appendix, and it is mostly right. For special relativity it covers the two postulates, time dilation, length contraction, E = mc² and flat spacetime. For general relativity it covers the equivalence principle, curved spacetime, light bending (the 1919 eclipse), gravitational time dilation (GPS), Mercury's orbit and gravitational waves. It doesn't mention black holes. A few things are a bit simplified:
   - It says special relativity can't handle acceleration. Special relativity can actually describe accelerating objects; what it can't describe is gravity. The real difference is flat vs curved spacetime.
   - It writes Einstein's field equations as Gᵘᵛ = 8πTᵘᵛ, which leaves out the constants G and c.
   - The airplane clock example only looks at speed. Higher up, gravity is weaker, which makes clocks run a little faster, so in real experiments the result depended on which direction the plane flew.

Overall, DeepSeek-R1 works locally with Ollama, `ollama ps` shows it running, and `ollama.chat()` gives back both the chain of thought and the answer. On a PC without a GPU it works, just slowly.

## Appendix - Concepts used in this problem

### 1. How the pieces fit together

```
 notebook: ollama.chat(), ollama.pull() ──┐
                                          ├─▶  Ollama server  ─▶  deepseek-r1:8b
 terminal: ollama ps, ollama --version ───┘   localhost:11434     in memory, on my CPU
```

Ollama is a program that runs open LLMs on my own computer. It starts a small server on `http://localhost:11434`. The `ollama` terminal command and the Python library are just two ways of sending requests to that server. After the model is downloaded, everything runs on my PC: no internet, no API key and no cost per question.

### 2. The model

| Concept | What it means | In my run |
|---|---|---|
| Parameters | The numbers (weights) the model learned during training. More parameters usually means better answers, but more memory and slower. | 8 billion |
| Name and tag | `model:tag`. The tag picks the size. A name without a tag means `:latest`. | `deepseek-r1:8b` |
| Distilled model | The full DeepSeek-R1 has 671B parameters, far too big for a PC. The smaller sizes are smaller models trained to copy how the big one reasons. | the 8b distilled version |
| Quantization | Storing each weight with fewer bits. 8 billion weights × 16 bits = 16 GB. Ollama stores them with about 4-5 bits each (some parts keep more), which gives about 5 GB. | 5.2 GB download |
| Token | A piece of a word. The model reads and writes text as tokens. | 2,193 tokens generated |

### 3. Memory, CPU and GPU

The whole model has to fit in memory. To generate every single token, the computer has to do math with all of the weights. A GPU does that math in parallel and has much faster memory, so it is many times faster than a CPU. Without a GPU the model still works, just slowly.

In my run: no NVIDIA GPU, so `ollama ps` showed **100% CPU** and the speed was **2.3 tokens per second**.

### 4. The commands I used

| Command | What it does |
|---|---|
| `ollama --version` | checks that Ollama is installed |
| `ollama.pull(model)` (= `ollama pull model`) | downloads the model, or checks it if it is already there |
| `ollama.generate(model, prompt="", keep_alive="30m")` | loads the model into memory without asking anything |
| `ollama ps` | lists the models that are loaded in memory right now |
| `ollama.chat(model, messages, think=True)` | asks a question and returns the thinking and the answer separately |

`keep_alive` is how long the model stays in memory after the last request (5 minutes by default). I used 30 minutes so it was still loaded for the question.

**How to read `ollama ps`:**

| Column | What it means | In my run |
|---|---|---|
| NAME | the model that is loaded | `deepseek-r1:8b` |
| SIZE | memory in use: the weights plus memory for the context, so more than the download | 5.9 GB |
| PROCESSOR | how much runs on the GPU vs the CPU; `100% GPU` is fastest | 100% CPU |
| CONTEXT | how many tokens the model can handle at once: my question plus everything it writes | 4096 (my run used 17 + 2,193 = 2,210) |
| RUNNER | the engine that runs the model | llamacpp (llama.cpp) |
| UNTIL | when Ollama will unload the model (`keep_alive`) | 29 minutes from now |

### 5. Reasoning models and the chain of thought

A normal model starts writing the answer right away. A **reasoning model** like DeepSeek-R1 first writes a **chain of thought**: it thinks through the problem step by step, plans the answer and checks itself, and only then writes the final answer. This can make answers better, but all the thinking is extra tokens, so it is slower.

With `think=True`, Ollama returns the two parts separately:

| Field | What it holds | In my run |
|---|---|---|
| `response.message.thinking` | the chain of thought | 4,174 characters |
| `response.message.content` | the final answer only | 6,366 characters |

Older Ollama versions put both parts into `content`, with the thinking between `<think>` and `</think>` tags.

### 6. The numbers in the response

| Field | What it means | In my run |
|---|---|---|
| `prompt_eval_count` | tokens in my question | 17 |
| `eval_count` | tokens the model generated (thinking + answer) | 2,193 |
| `eval_duration` | time spent generating, in nanoseconds (1 s = 10⁹ ns) | about 950 s |
| speed | `eval_count / (eval_duration / 10⁹)` | 2.3 tokens/s |
| time | whole call, measured with `time.time()` | 981.5 s (about 16 min) |

### 7. The question: Special vs General relativity

The summary I used to check the model's answer:

| | Special relativity (1905) | General relativity (1915) |
|---|---|---|
| Spacetime | flat | curved by mass and energy |
| Gravity | not included | gravity *is* the curvature of spacetime |
| Main ideas | the laws of physics are the same for observers moving at constant speed; the speed of light is the same for every observer | equivalence principle: gravity and acceleration feel the same |
| Results | time dilation, length contraction, E = mc² | light bending around the Sun, clocks running slower in strong gravity (GPS corrects for it), black holes, gravitational waves |

**The main difference is flat vs curved spacetime.** Special relativity can describe moving and even accelerating objects, but not gravity. Special relativity is the special case of general relativity when gravity can be ignored.